# Module 3 Post-Fire Recovery

Vegetation trajectory from 2021 through 2025, stratified by burn severity.

The core method is curve fitting rather than trend-line drawing. An asymptotic
exponential is fitted to each severity class, giving two parameters that matter
independently:

- **Rate**: how fast recovery proceeds
- **Asymptote**: the ceiling it approaches, which may be well below pre-fire

The asymptote is the parameter people skip and the one that carries the
ecological content. Fast recovery to a low ceiling is grass and shrub
establishment. Slow recovery to a high ceiling is forest returning. In NDVI these
look similar for the first several years, and a linear trend cannot tell them
apart at all.

**Stratification by severity is not optional.** A whole-fire mean averages ground
that barely burned with ground that lost its entire canopy and describes
neither.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import daear_toolkit as dt
from daear_toolkit import data_access, indicators, viz
from daear_toolkit import fire_access as fa
from daear_toolkit import fire_indicators as fi

REGION = dt.POUDRE_CAMERON_PEAK
BBOX = REGION.bbox

# Severity from Module 2. Re-run 02 first if this file is missing.
severity = xr.open_dataarray("../outputs/02_severity_class.nc")
perimeters = fa.get_mtbs_perimeters(BBOX, year=2020, min_acres=1000)
print("Severity classes present:", np.unique(severity.values[np.isfinite(severity.values)]))

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\gekek\\Documents\\Daear-Consulting demos\\daear-earth-intelligence\\wildfire-landscape-intelligence\\outputs\\02_severity_class.nc'

## Annual NDVI series

One consistent late-summer composite per year, so that phenology is not mistaken
for trend. Mid-July to mid-August matches the window used in Modules 1 and 2.

Two pre-fire years (2018, 2019) establish the baseline; 2020 is excluded because
the fire was burning through it.

In [ ]:
def annual_ndvi(year, start="07-15", end="08-31"):
    scene = data_access.get_optical_scene(BBOX, start=f"{year}-{start}", end=f"{year}-{end}", max_cloud_pct=15)
    return indicators.ndvi(scene)

PRE_YEARS = [2018, 2019]
POST_YEARS = [2021, 2022, 2023, 2024, 2025]

ndvi_pre = {y: annual_ndvi(y) for y in PRE_YEARS}
ndvi_post = {y: annual_ndvi(y) for y in POST_YEARS}

prefire_ndvi = sum(ndvi_pre.values()) / len(ndvi_pre)

fig, axes = plt.subplots(1, len(POST_YEARS) + 1, figsize=(3.1 * (len(POST_YEARS) + 1), 3.6))
viz.plot_raster(prefire_ndvi, title="Pre-fire NDVI (2018-2019)", ax=axes[0], cmap="YlGn", vmin=0, vmax=0.8)
for ax, y in zip(axes[1:], POST_YEARS):
    viz.plot_raster(ndvi_post[y], title=f"NDVI {y}", ax=ax, cmap="YlGn", vmin=0, vmax=0.8)
for ax in axes:
    perimeters.boundary.plot(ax=ax, color="black", lw=0.6)
plt.tight_layout()
plt.savefig("../outputs/03_ndvi_series.png", dpi=150)
plt.show()

## Recovery trajectories by severity class

In [ ]:
recovery = fi.recovery_by_severity(ndvi_post, severity, prefire_ndvi)
recovery.to_csv("../outputs/03_recovery_by_severity.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
palette = {"unburned/regrowth": "#2a7f4f", "low": "#9dbf3f", "moderate": "#e08a2e", "high": "#b5443a"}

for label, g in recovery.groupby("severity"):
    axes[0].plot(g["year"], g["mean_ndvi"], "o-", color=palette.get(label), label=label)
    axes[1].plot(g["year"], g["recovery_ratio"], "o-", color=palette.get(label), label=label)

axes[0].axhline(float(prefire_ndvi.mean()), color="grey", ls="--", lw=1, label="pre-fire mean")
axes[0].set_ylabel("mean NDVI"); axes[0].set_title("Absolute NDVI by severity class")
axes[1].axhline(1.0, color="grey", ls="--", lw=1)
axes[1].set_ylabel("recovery ratio (post / pre)"); axes[1].set_title("Fraction of pre-fire NDVI recovered")
for ax in axes:
    ax.legend(fontsize=8); ax.set_xlabel("year")
plt.tight_layout()
plt.savefig("../outputs/03_recovery_trajectories.png", dpi=150)
plt.show()

recovery.pivot(index="year", columns="severity", values="recovery_ratio").round(3)

## Fit the curves

Asymptotic exponential per severity class, reporting rate, ceiling, half-life,
and fit quality.

Read the two parameters together. A high-severity class showing a fast rate to a
0.5 asymptote is not recovering forest, it is converting to grass and shrub,
which is exactly the type conversion that concerns managers in the Southern
Rockies after high-severity fire. Reporting only "recovery rate" would hide
that.

In [ ]:
fits = []
for label, g in recovery.groupby("severity"):
    g = g.sort_values("year")
    fit = fi.fit_recovery_curve(g["year"].values, g["mean_ndvi"].values)
    fit["severity"] = label
    fit["prefire_ndvi"] = round(float(g["prefire_ndvi"].iloc[0]), 4)
    fit["ceiling_vs_prefire"] = round(fit["asymptote"] / fit["prefire_ndvi"], 3) if fit["converged"] else np.nan
    fits.append(fit)

fit_df = pd.DataFrame(fits)[
    ["severity", "converged", "asymptote", "prefire_ndvi", "ceiling_vs_prefire", "rate", "half_life_years", "r_squared", "n"]
].round(3)
fit_df.to_csv("../outputs/03_recovery_fits.csv", index=False)
print(fit_df)

print("\nOnly 5 post-fire years. An asymptotic fit to 5 points constrains the RATE")
print("reasonably and the CEILING poorly, the asymptote is an extrapolation beyond")
print("the data, and its standard error should be quoted alongside it. Treat ceiling")
print("estimates as provisional until the series is 8-10 years long.")
for f in fits:
    if f.get("converged") and f.get("param_se"):
        print(f"  {f['severity']:<20} asymptote {f['asymptote']:.3f} +/- {f['param_se'][0]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
t_smooth = np.linspace(0, 12, 120)
for f in fits:
    if not f["converged"]:
        continue
    g = recovery[recovery["severity"] == f["severity"]].sort_values("year")
    ax.plot(g["year"], g["mean_ndvi"], "o", color=palette.get(f["severity"]), ms=6)
    ax.plot(2021 + t_smooth, fi._recovery_model(t_smooth, f["asymptote"], f["rate"], f["initial"]),
            color=palette.get(f["severity"]), lw=2,
            label=f"{f['severity']} (ceiling {f['asymptote']:.2f}, t50 {f['half_life_years']:.1f}y)")
    ax.axhline(f["prefire_ndvi"], color=palette.get(f["severity"]), ls=":", lw=1, alpha=0.6)

ax.axvspan(2026, 2033, color="grey", alpha=0.12)
ax.text(2029.5, ax.get_ylim()[0] + 0.02, "extrapolation", ha="center", fontsize=9, color="grey")
ax.set_xlabel("year"); ax.set_ylabel("mean NDVI")
ax.set_title("Fitted recovery curves (dotted lines = pre-fire NDVI per class)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("../outputs/03_recovery_curves.png", dpi=150)
plt.show()

## Where is recovery lagging?

Class means hide spatial structure. A per-pixel recovery ratio identifies the
specific ground that is not coming back, which is where replanting and erosion
control get targeted.

The layer to hand a manager is not "low recovery" it is **high severity AND low
recovery**, since low recovery on ground that barely burned means something else
is going on (aspect, soil, grazing) and does not call for post-fire
intervention.

In [ ]:
latest = ndvi_post[max(POST_YEARS)]

# Clipped at 1.5 rather than 1.0 on purpose: post-fire herbaceous flushes
# genuinely exceed pre-fire NDVI in the first seasons, and that overshoot is a
# real signal (grass, not conifer) that clipping to 1.0 would hide. The 0.05
# floor guards against dividing by a near-zero pre-fire baseline.
recovery_ratio = (latest / prefire_ndvi.where(prefire_ndvi > 0.05)).clip(0, 1.5)

lagging = (recovery_ratio < 0.6) & (severity == 3)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
viz.plot_raster(recovery_ratio, title=f"Recovery ratio, {max(POST_YEARS)} vs pre-fire", ax=axes[0], cmap="RdYlGn", vmin=0, vmax=1.2)
viz.plot_raster(lagging.where(lagging), title="High severity AND recovery < 60%", ax=axes[1], cmap="Reds")
for ax in axes:
    perimeters.boundary.plot(ax=ax, color="black", lw=0.7)
plt.tight_layout()
plt.savefig("../outputs/03_lagging_recovery.png", dpi=150)
plt.show()

recovery_ratio.to_netcdf("../outputs/03_recovery_ratio.nc")
print(f"Area high-severity and under 60% recovered: {float(lagging.mean()):.1%} of the AOI")
print(f"  ({int(lagging.sum()) * 0.04:.0f} hectares at 20 m resolution)")

## Summary

Severity-stratified recovery trajectories, fitted curves with ceilings and
half-lives, and a spatial layer identifying where high-severity ground is
recovering slowly.

`03_recovery_ratio.nc` feeds Module 4's risk assessment, and the same pattern is
what `soil-watershed-intelligence` Module 4 adapts to treated-vs-untreated
restoration comparison.

**Limitations, most important first:**

1. **NDVI recovery is not ecological recovery.** A slope flushing with cheatgrass
   scores well here. This is the metric's central failure mode because
   species composition needs field plots, and the highest-value pairing for this
   analysis is monitoring transects rather than more satellite data.
2. **Five post-fire years constrain the rate, not the ceiling.** Asymptotes are
   extrapolations; quote their standard errors and revisit at 8–10 years.
3. **No control for climate.** A wet year lifts every class simultaneously.
   Differencing against unburned control ground inside the same AOI is the
   approach used in `soil-watershed-intelligence` Module 4 and would separate
   recovery from favourable weather, and is the obvious next iteration here.
4. **Severity class boundaries are borrowed**, inherited from Module 2's
   uncalibrated Key & Benson thresholds. Class-level conclusions inherit that
   uncertainty.